In [ ]:
from ingest import load_data
documents = load_data()

In [2]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [12]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [57]:
data_gen_instructions = """
You emulate a user who is searching for quick recipes based on their time or 
available groceries or nutrition and dietary preferences.
Formulate 2 questions this user might ask to get particular recipe suggested. 
Questions don't need to strictly correspond to the recipe.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [58]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [59]:
doc = documents[100]

In [60]:
import json
user_prompt = json.dumps(doc)

In [61]:
doc

{'id': '200',
 'recipe_name': 'Cherry Chicken',
 'total_time': '45 mins',
 'servings': '4',
 'ingredients': '3 tablespoons vegetable oil, 1 (4 pound) whole chicken, cut into 8 pieces,   salt and pepper to taste, ½ cup all-purpose flour for dusting, 1 (15 ounce) can pitted dark cherries packed in water, ½ cup white sugar, 1 tablespoon cornstarch, 1  orange - with peel, quartered and thinly sliced, ½ cup slivered almonds, toasted',
 'directions': 'Heat oil in a large skillet over medium-high heat. Season chicken with salt and pepper, then coat with flour. Fry in hot oil until browned, turning as needed. Reduce heat to medium, cover, and cook until meat is tender and juices run clear, about 25 minutes.\nRemove chicken from the skillet; pour off all but 1/4 cup drippings. Return the skillet to medium heat and stir in cherries, reserving some cherry liquid for later. Stir in sugar and bring to a boil. Dissolve cornstarch in reserved cherry liquid, then stir into the skillet. Cook, stirring 

In [62]:
user_prompt

'{"id": "200", "recipe_name": "Cherry Chicken", "total_time": "45 mins", "servings": "4", "ingredients": "3 tablespoons vegetable oil, 1 (4 pound) whole chicken, cut into 8 pieces,   salt and pepper to taste, \\u00bd cup all-purpose flour for dusting, 1 (15 ounce) can pitted dark cherries packed in water, \\u00bd cup white sugar, 1 tablespoon cornstarch, 1  orange - with peel, quartered and thinly sliced, \\u00bd cup slivered almonds, toasted", "directions": "Heat oil in a large skillet over medium-high heat. Season chicken with salt and pepper, then coat with flour. Fry in hot oil until browned, turning as needed. Reduce heat to medium, cover, and cook until meat is tender and juices run clear, about 25 minutes.\\nRemove chicken from the skillet; pour off all but 1/4 cup drippings. Return the skillet to medium heat and stir in cherries, reserving some cherry liquid for later. Stir in sugar and bring to a boil. Dissolve cornstarch in reserved cherry liquid, then stir into the skillet. 

In [63]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [64]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [65]:
response.output_parsed.questions

['Got about 45 minutes and a whole chicken — what’s an easy dinner recipe that uses cherries or some kind of fruit sauce?',
 'What’s a savory-sweet chicken recipe with cherries and oranges that I can make for 4 people?']

In [66]:
from evaluation_utils import llm_structured

In [67]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What’s a good easy dinner with chicken and cherries that takes around 45 minutes?', 'Any sweet-and-savory chicken recipe with fruit and almonds for 4 people?']


In [68]:
usage

ResponseUsage(input_tokens=519, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=46, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=565)

In [69]:
from evaluation_utils import calc_price

In [70]:
calc_price(usage)

{'input_cost': 0.00038925000000000006,
 'output_cost': 0.000207,
 'total_cost': 0.00059625}

In [71]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'What’s a good easy dinner with chicken and cherries that takes around 45 minutes?',
  'document': '200'},
 {'question': 'Any sweet-and-savory chicken recipe with fruit and almonds for 4 people?',
  'document': '200'}]

In [14]:
import pandas as pd

In [15]:
pd.DataFrame(records)

,question,document
0,"How is the capstone homework graded, and what ...",0d200c8c58
1,What are the point values for answering the qu...,0d200c8c58
2,How many points can I earn from one homework i...,0d200c8c58
3,Does the homework score depend on correct answ...,0d200c8c58
4,What tasks should I complete to maximize my ho...,0d200c8c58


In [74]:
from evaluation_utils import llm_structured_retry

In [75]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [77]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/598 [00:00<?, ?it/s]

In [24]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

395

In [27]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [28]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.05718675000000001

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.05718675000000001

In [30]:
df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [33]:
len(df_ground_truth)

395